In [ ]:
import pandas as pd
import numpy as np
import os
import pickle
from sklearn.preprocessing import StandardScaler


combined_df = pd.DataFrame()

file_paths = [
        './hist_data_round_2/prices_round_2_day_-1.csv',
        './hist_data_round_2/prices_round_2_day_0.csv',
        './hist_data_round_2/prices_round_2_day_1.csv',
        './hist_data_round_3/prices_round_3_day_2.csv',
        # '../hist_data_round_4/prices_round_4_day_3.csv',
    ]

excluded_products = ['RAINFOREST_RESIN', 'SQUID_INK', 
                         'CROISSANTS', 'DJEMBES', 'JAMS', 
                         'PICNIC_BASKET1', 'PICNIC_BASKET2',
                         'VOLCANIC_ROCK_VOUCHER_9750',
                         'VOLCANIC_ROCK_VOUCHER_9500',
                         'VOLCANIC_ROCK',
                         'VOLCANIC_ROCK_VOUCHER_10250',
                         'VOLCANIC_ROCK_VOUCHER_10000',
                         'VOLCANIC_ROCK_VOUCHER_10500']
# looking at KELP



# Read and combine all CSV files
for file_path in file_paths:
    if os.path.exists(file_path):
        print(f"Reading file: {file_path}")
        df = pd.read_csv(file_path, sep=';')
        combined_df = pd.concat([combined_df, df], ignore_index=True)
    else:
        print(f"Warning: File not found: {file_path}")

print(f"Combined data shape before filtering: {combined_df.shape}")

if excluded_products:
    original_count = len(combined_df)
    combined_df = combined_df[~combined_df['product'].isin(excluded_products)]
    filtered_count = len(combined_df)
    print(f"Filtered out {original_count - filtered_count} rows with products: {excluded_products}")
    print(f"Combined data shape after filtering: {combined_df.shape}")

# create new features
def weighted_mean(window):
    weights = np.arange(1, len(window) + 1)  # Assign weights (1, 2, 3, ...)
    return np.dot(window, weights) / weights.sum()

def mark_peaks_troughs(index, window = 100, sharp = 0.00001):
    # Define the range for left and right neighbors
    left_range = combined_df['weighted_avg'].iloc[max(0, index - window):index]
    right_range = combined_df['weighted_avg'].iloc[index + 1:min(len(combined_df), index + window +1 - 50)]

    current_price = combined_df['weighted_avg'].iloc[index]
    # print(current_price)
    # Check for peak
    if current_price >= left_range.max()  and current_price >= right_range.max() and current_price >= left_range.mean() * (1+sharp)\
        and current_price >= right_range.mean() * (1+sharp):

        return 1  # Peak
    # Check for trough
    elif current_price <= left_range.min() and current_price <= right_range.min()\
        and current_price <= left_range.mean() * (1-sharp) and current_price <= right_range.mean() * (1-sharp):

        return 0  # Trough
    return None  # Neither

def replace_with_nan(arr):
    """
    Replaces values in a NumPy array with NaN if they are the same as the next valid value.

    Args:
        arr (np.ndarray): The input NumPy array containing 0s, 1s, and NaNs.

    Returns:
        np.ndarray: A new NumPy array with the replacements made.
    """
    arr = np.copy(arr)  # Operate on a copy to avoid modifying the original
    valid_indices = np.where(~np.isnan(arr))[0]  # Indices of valid (non-NaN) values

    if len(valid_indices) < 2:
        return arr  # Nothing to compare if there are fewer than 2 valid values

    for i in range(len(valid_indices) - 1):
        current_index = valid_indices[i]
        next_index = valid_indices[i+1]

        if arr[current_index] == arr[next_index]:
            arr[current_index] = np.nan

    return arr
combined_df['new_bid_volume'] = combined_df['bid_volume_1'].fillna(0) + combined_df['bid_volume_2'].fillna(0) * 0.8
combined_df['new_ask_volume'] = combined_df['ask_volume_1'].fillna(0) + combined_df['ask_volume_2'].fillna(0) * 0.8
combined_df['bidask'] = combined_df['new_bid_volume'] / combined_df ['new_ask_volume']
combined_df['weighted_avg'] = combined_df['mid_price'].rolling(150).apply(weighted_mean, raw=False)


Reading file: ./hist_data_round_2/prices_round_2_day_-1.csv
Reading file: ./hist_data_round_2/prices_round_2_day_0.csv
Reading file: ./hist_data_round_2/prices_round_2_day_1.csv
Reading file: ./hist_data_round_3/prices_round_3_day_2.csv
Combined data shape before filtering: (380000, 17)
Filtered out 340000 rows with products: ['RAINFOREST_RESIN', 'SQUID_INK', 'CROISSANTS', 'DJEMBES', 'JAMS', 'PICNIC_BASKET1', 'PICNIC_BASKET2', 'VOLCANIC_ROCK_VOUCHER_9750', 'VOLCANIC_ROCK_VOUCHER_9500', 'VOLCANIC_ROCK', 'VOLCANIC_ROCK_VOUCHER_10250', 'VOLCANIC_ROCK_VOUCHER_10000', 'VOLCANIC_ROCK_VOUCHER_10500']
Combined data shape after filtering: (40000, 17)


In [3]:
delta = combined_df['mid_price'].diff()
gain = (delta.where(delta >0, 0)).rolling(window=30).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=30).mean()
rs = gain / loss
rsi = 100 - (100 / (1 + rs))
combined_df['RSI'] = rsi.fillna(method='ffill')

C:\Users\Casen\AppData\Local\Temp\ipykernel_25528\428733293.py:6: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  combined_df['RSI'] = rsi.fillna(method='ffill')


In [4]:
short_ema = combined_df['mid_price'].rolling(15).apply(weighted_mean, raw=False)

# Calculate the long-term EMA (26 periods)
long_ema = combined_df['mid_price'].rolling(30).apply(weighted_mean, raw=False)

# Calculate the MACD line
combined_df['MACD'] = short_ema - long_ema

In [5]:
combined_df['peak_trough'] = [mark_peaks_troughs(i) for i in range(len(combined_df))]
combined_df['peak_trough'] = replace_with_nan(combined_df['peak_trough'])
combined_df['peak_trough'] = combined_df['peak_trough'].fillna(method = 'bfill')

C:\Users\Casen\AppData\Local\Temp\ipykernel_25528\751724447.py:3: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  combined_df['peak_trough'] = combined_df['peak_trough'].fillna(method = 'bfill')


In [6]:
reduced_df  =combined_df[['weighted_avg', 'bidask', 'RSI', 'MACD', 'peak_trough']]

In [7]:
trim_df = reduced_df.dropna()


# valid_rows = df.notna().all(axis=1)
# first_valid_idx = valid_rows.idxmax()
# last_valid_idx = valid_rows[::-1].idxmax()
# # Find the last valid index after dropping NaNs

# trim_df = reduced_df.iloc[first_valid_idx:last_valid_idx+1]
# trim_df

In [8]:
window_size = 30
max_windows = len(trim_df) - window_size 
label = trim_df['peak_trough'].copy()
X_windows_3d = np.zeros((max_windows, window_size, 4))
y_labels = np.zeros(max_windows)

for i in range(max_windows):

    features_window = trim_df[['weighted_avg', 'bidask', 'MACD', 'RSI']].iloc[i:i + window_size].values

    # Store in our 3D array
    X_windows_3d[i] = features_window

    # Create label: 1 if price goes up, 0 if it goes down or stays the same
    y_labels[i] = label.iloc[i + window_size - 1]  # Last price in window (50th)

print(f"X_windows_3d shape: {X_windows_3d.shape}")
print(f"y_labels shape: {y_labels.shape}")


X_windows_3d shape: (39875, 30, 4)
y_labels shape: (39875,)


In [9]:

mean_vals = X_windows_3d[:, :, :3].mean(axis=1, keepdims=True)  # Mean of channels 0 and 1
std_vals = X_windows_3d[:, :, :3].std(axis=1, keepdims=True)    # Std of channels 0 and 1
epsilon = 1e-10
normalized_price_bidask = (X_windows_3d[:, :, :3] - mean_vals) / (std_vals + epsilon)

# 2. Min-Max Scale RSI (channel 2)
rsi_data = X_windows_3d[:, :, 3]  # Extract RSI data
rsi_scaled = rsi_data /100
rsi_scaled = np.expand_dims(rsi_scaled, axis=2)


# 3. Combine the preprocessed channels
normalised = np.concatenate((normalized_price_bidask, rsi_scaled), axis=2)





In [10]:
np.save(f"new2_traininginput_3d.npy", normalised)

np.save("new2_traininglabel_3d.npy", y_labels)
